# Lesson 8.3 — Reading the Integrated System: Guarantees and Boundaries

The system guarantees accounting, completion, graceful degradation, and localised faults — and explicitly does NOT guarantee completeness, diagnosis, or optimal sequencing.

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, Fruit, model_perception,
    understand, to_configuration, recover, harvest_row)
W = GreenhouseWorld.demo_row(n=6, seed=1)
DETS = model_perception(W, rng=np.random.default_rng(7))
TGT = understand(DETS, W)[1]            # nearest ripe reachable target
kick = lambda t, j: 60.0 if (j == 0 and t > 0.3) else 0.0
checks = []
all_ids = {f.fid for f in W.fruit if f.ripe}
vxy = next(d['xy'] for d in DETS if d['id'] == understand(DETS, W)[1]['id'])
victim = understand(DETS, W)[1]['id']
r = harvest_row(W, inject={victim: dict(obstacle=(vxy, 0.25))})
print('report -> harvested=%s skipped=%s complete=%s' % (r['harvested'], r['skipped'], r['complete']))


report -> harvested=['F5', 'F0', 'F2', 'F1'] skipped=['F3'] complete=True


### Guarantee 1: ACCOUNTING -- every ripe fruit is harvested or skipped, none lost

In [2]:
accounted = set(r['harvested']) | set(r['skipped'])
checks.append(all_ids <= accounted)            # no ripe fruit silently lost
checks.append(set(r['harvested']).isdisjoint(set(r['skipped'])))   # exactly one bucket
# Guarantee 2: COMPLETION
checks.append(r['complete'] is True)
# Guarantee 3: LOCALISED FAULTS -- every skip carries a fault code
for p in r['picks']:
    if p['outcome'] == 'skipped':
        checks.append(p.get('fault') is not None)


### Boundary: a skipped fruit is recorded, NOT re-picked endlessly (best-effort, not complete)

In [3]:
print('skipped (recorded, not retried):', r['skipped'])
checks.append(victim in r['skipped'] and victim not in r['harvested'])
# best-effort: harvested is a subset of ripe fruit, not necessarily all of it
checks.append(set(r['harvested']) <= all_ids and len(r['harvested']) < len(all_ids))
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


skipped (recorded, not retried): ['F3']
6/6 checks passed.
All checks passed.
